In [1]:
import RPi.GPIO as GPIO
import time

TRIG = 20
ECHO = 21

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BCM)

GPIO.setup(TRIG, GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)

GPIO.output(TRIG, GPIO.LOW)

def measure_distance(timeout=0.03):
    GPIO.output(TRIG, GPIO.LOW)
    time.sleep(0.000002)

    GPIO.output(TRIG, GPIO.HIGH)
    time.sleep(0.00001)
    GPIO.output(TRIG, GPIO.LOW)

    deadline = time.monotonic() + timeout

    while GPIO.input(ECHO) == GPIO.LOW:
        if time.monotonic() > deadline:
            return None

    start = time.monotonic()
    deadline = start + timeout

    while GPIO.input(ECHO) == GPIO.HIGH:
        if time.monotonic() > deadline:
            return None

    end = time.monotonic()

    return round((end - start) * 34300 / 2, 1)

In [2]:
try:
    for i in range(15):
        distance = measure_distance()
        print(f"Distance: {distance} cm")
        time.sleep(0.2)
finally:
    GPIO.cleanup()

Distance: 35.8 cm
Distance: 35.8 cm
Distance: 35.8 cm
Distance: 35.8 cm
Distance: 35.8 cm
Distance: 35.8 cm
Distance: 11.2 cm
Distance: 7.5 cm
Distance: 5.6 cm
Distance: 11.5 cm
Distance: 12.9 cm
Distance: 16.3 cm
Distance: 16.3 cm
Distance: 11.6 cm
Distance: 5.6 cm


In [3]:
def ultrasonic_decision(distance):
    if distance is None:
        return "STOP"
    elif distance < 40:
        return "AVOID"
    else:
        return "FORWARD"

In [6]:
test_distances = [60, 45, 40, 35.8, 20, 10, None]

for distance in test_distances:
    action = ultrasonic_decision(distance)
    print(f"Distance: {distance} cm -> {action}")

Distance: 60 cm -> FORWARD
Distance: 45 cm -> FORWARD
Distance: 40 cm -> FORWARD
Distance: 35.8 cm -> AVOID
Distance: 20 cm -> AVOID
Distance: 10 cm -> AVOID
Distance: None cm -> STOP


In [8]:
from LOBOROBOT import LOBOROBOT

robot = LOBOROBOT()

robot.t_stop(0)

print("Robot initialized successfully!")

Robot initialized successfully!


In [9]:
import time

THRESHOLD = 20
SPEED = 25
TEST_DURATION = 10

def distance_action(distance):
    if distance is None:
        return "STOP"
    if distance < THRESHOLD:
        return "STOP"
    return "FORWARD"

start_time = time.monotonic()

try:
    robot.t_stop(0)

    while time.monotonic() - start_time < TEST_DURATION:
        distance = measure_distance()
        action = distance_action(distance)

        print(f"Distance: {distance} cm | Action: {action}")

        if action == "FORWARD":
            robot.t_up(SPEED, 0.1)
        else:
            robot.t_stop(0)
            time.sleep(0.1)

        robot.t_stop(0)

except KeyboardInterrupt:
    print("Interrupted.")

finally:
    robot.t_stop(0)
    GPIO.cleanup()
    print("Experiment 05 test completed.")

Experiment 05 test completed.


RuntimeError: The GPIO channel has not been set up as an OUTPUT

In [10]:
import RPi.GPIO as GPIO
import time
from LOBOROBOT import LOBOROBOT

# 1. Initialize robot
robot = LOBOROBOT()
robot.t_stop(0)

# 2. Configure ultrasonic sensor
TRIG = 20
ECHO = 21

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BCM)

GPIO.setup(TRIG, GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)

GPIO.output(TRIG, GPIO.LOW)

print("Robot initialized successfully!")
print("Ultrasonic sensor initialized successfully!")

Robot initialized successfully!
Ultrasonic sensor initialized successfully!


In [11]:
print("GPIO mode:", GPIO.getmode())
print("TRIG function:", GPIO.gpio_function(TRIG))
print("ECHO function:", GPIO.gpio_function(ECHO))

GPIO mode: 11
TRIG function: 0
ECHO function: 1


In [12]:
for i in range(5):
    distance = measure_distance()
    print(f"Distance: {distance} cm")
    time.sleep(0.2)

Distance: 70.2 cm
Distance: 76.1 cm
Distance: None cm
Distance: None cm
Distance: 66.3 cm


In [13]:
import time

THRESHOLD = 40
SPEED = 25
TEST_DURATION = 8

def distance_action(distance):
    if distance is None:
        return "STOP"
    elif distance < THRESHOLD:
        return "STOP"
    else:
        return "FORWARD"

start_time = time.monotonic()

try:
    robot.t_stop(0)

    while time.monotonic() - start_time < TEST_DURATION:
        distance = measure_distance()
        action = distance_action(distance)

        print(f"Distance: {distance} cm | Action: {action}")

        if action == "FORWARD":
            robot.t_up(SPEED, 0.1)
        else:
            robot.t_stop(0)
            time.sleep(0.1)

        robot.t_stop(0)

except KeyboardInterrupt:
    print("Interrupted.")

finally:
    try:
        robot.t_stop(0)
    finally:
        GPIO.cleanup()

    print("Experiment 05 test completed.")

Distance: 56.4 cm | Action: FORWARD
Distance: 56.6 cm | Action: FORWARD
Distance: 61.6 cm | Action: FORWARD
Distance: 35.8 cm | Action: STOP
Distance: 57.4 cm | Action: FORWARD
Distance: 56.6 cm | Action: FORWARD
Distance: 35.8 cm | Action: STOP
Distance: 56.9 cm | Action: FORWARD
Distance: 36.1 cm | Action: STOP
Distance: 35.9 cm | Action: STOP
Distance: 12.9 cm | Action: STOP
Distance: 5.6 cm | Action: STOP
Distance: 7.9 cm | Action: STOP
Distance: 12.6 cm | Action: STOP
Distance: 16.3 cm | Action: STOP
Distance: 37.9 cm | Action: STOP
Distance: 13.9 cm | Action: STOP
Distance: 12.6 cm | Action: STOP
Distance: 11.2 cm | Action: STOP
Distance: 10.6 cm | Action: STOP
Distance: 8.9 cm | Action: STOP
Distance: 10.2 cm | Action: STOP
Distance: 7.1 cm | Action: STOP
Distance: 5.5 cm | Action: STOP
Distance: 3.6 cm | Action: STOP
Distance: 4.9 cm | Action: STOP
Distance: 7.5 cm | Action: STOP
Distance: 8.5 cm | Action: STOP
Distance: 8.8 cm | Action: STOP
Distance: 53.1 cm | Action: FORWARD

In [14]:
import statistics
import time

def filtered_distance(samples=3):
    readings = []

    for _ in range(samples):
        d = measure_distance()

        if d is not None and 2 <= d <= 400:
            readings.append(d)

        time.sleep(0.06)

    if not readings:
        return None

    return round(statistics.median(readings), 1)

In [15]:
import RPi.GPIO as GPIO

# Reinitialize ultrasonic GPIO after previous cleanup
GPIO.setmode(GPIO.BCM)
GPIO.setup(TRIG, GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)
GPIO.output(TRIG, GPIO.LOW)

for i in range(10):
    distance = filtered_distance()
    print(f"Filtered distance: {distance} cm")

Filtered distance: 56.6 cm
Filtered distance: 36.9 cm
Filtered distance: 70.2 cm
Filtered distance: 63.2 cm
Filtered distance: None cm
Filtered distance: 16.9 cm
Filtered distance: 12.6 cm
Filtered distance: 10.5 cm
Filtered distance: 9.5 cm
Filtered distance: 12.3 cm


In [16]:
import RPi.GPIO as GPIO
import time

# robot should already be initialized
robot.t_stop(0)

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BCM)

TRIG = 20
ECHO = 21

GPIO.setup(TRIG, GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)
GPIO.output(TRIG, GPIO.LOW)

In [17]:
THRESHOLD = 40
SPEED = 25

def avoid_obstacle(robot):
    # Stop before changing direction
    robot.t_stop(0)

    try:
        # Move backward briefly
        robot.t_down(SPEED, 0.3)
        robot.t_stop(0)

        time.sleep(0.1)

        # Turn right briefly
        robot.turnRight(SPEED, 0.4)

    finally:
        robot.t_stop(0)


try:
    robot.t_stop(0)

    for i in range(5):
        distance = measure_distance()

        print(f"Distance: {distance} cm")

        if distance is None:
            print("Sensor timeout -> STOP")
            robot.t_stop(0)

        elif distance < THRESHOLD:
            print("Obstacle detected -> AVOID")
            avoid_obstacle(robot)

        else:
            print("Clear path -> FORWARD")
            robot.t_up(SPEED, 0.2)
            robot.t_stop(0)

        time.sleep(0.2)

finally:
    try:
        robot.t_stop(0)
    finally:
        GPIO.cleanup()

    print("Experiment 05 completed.")

Distance: 64.3 cm
Clear path -> FORWARD
Experiment 05 completed.


RuntimeError: The GPIO channel has not been set up as an OUTPUT

In [18]:
import RPi.GPIO as GPIO
import time
from LOBOROBOT import LOBOROBOT

# Initialize motor controller first
robot = LOBOROBOT()
robot.t_stop(0)

# Initialize ultrasonic sensor second
TRIG = 20
ECHO = 21

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BCM)

GPIO.setup(TRIG, GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)

GPIO.output(TRIG, GPIO.LOW)

print("Robot and ultrasonic sensor initialized.")

Robot and ultrasonic sensor initialized.


In [19]:
try:
    robot.t_up(25, 0.2)
finally:
    robot.t_stop(0)

print("Motor test completed.")

Motor test completed.


In [21]:
import RPi.GPIO as GPIO
import time
from LOBOROBOT import LOBOROBOT

TRIG = 20
ECHO = 21

SPEED = 25
THRESHOLD = 40

robot = None

try:
    # 1. Initialize robot
    robot = LOBOROBOT()
    robot.t_stop(0)

    # 2. Initialize ultrasonic sensor
    GPIO.setwarnings(False)
    GPIO.setmode(GPIO.BCM)

    GPIO.setup(TRIG, GPIO.OUT)
    GPIO.setup(ECHO, GPIO.IN)
    GPIO.output(TRIG, GPIO.LOW)

    print("Experiment 05 started!")

    # 3. Run five test cycles
    for i in range(5):

        distance = measure_distance()
        print(f"Distance: {distance} cm")

        if distance is None:
            print("Sensor timeout -> STOP")
            robot.t_stop(0)

        elif distance < THRESHOLD:
            print("Obstacle detected -> AVOID")

            robot.t_stop(0)

            # Reverse
            robot.t_down(SPEED, 0.3)
            robot.t_stop(0)

            time.sleep(0.1)

            # Turn right
            robot.turnRight(SPEED, 0.4)
            robot.t_stop(0)

        else:
            print("Clear path -> FORWARD")

            robot.t_up(SPEED, 0.2)
            robot.t_stop(0)

        time.sleep(0.2)

except KeyboardInterrupt:
    print("Experiment interrupted.")

finally:
    try:
        if robot is not None:
            robot.t_stop(0)
    finally:
        GPIO.cleanup()

    print("Experiment 05 test ended.")

Experiment 05 started!
Distance: 35.4 cm
Obstacle detected -> AVOID
Distance: 6.5 cm
Obstacle detected -> AVOID
Distance: 14.7 cm
Obstacle detected -> AVOID
Distance: 35.5 cm
Obstacle detected -> AVOID
Distance: 35.5 cm
Obstacle detected -> AVOID
Experiment 05 test ended.
